In [ ]:
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import deque
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from sklearn.preprocessing import MinMaxScaler
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True


# Load and normalize data

In [ ]:
# Load and normalize data
res_df = pd.read_excel("hourly_data.xlsx") #Moroccan data in hourly intervals
com_df = pd.read_excel("commercial_data.xlsx")
com_df['Commercial_load'] = com_df['Commercial_load'] + 250000 # offset to match Moroccan data ranges

loads_df = pd.DataFrame({
    'Residential_load':res_df['Total Power Consumption'][:192*3].values,
    'Commercial_load':com_df['Commercial_load'][:192*3].values
})

In [ ]:
# Normalize data
min_max_scaler = MinMaxScaler()
grad_scaler = torch.amp.GradScaler('cuda')
res_max = 700000
com_max = 600000
res_load_data = (loads_df['Residential_load'].values).reshape(-1, 1)
res_norm_load = min_max_scaler.fit_transform(res_load_data)

com_load_data = (loads_df['Commercial_load'].values).reshape(-1, 1)
com_norm_load = min_max_scaler.fit_transform(com_load_data)

util_res_load_data = (loads_df['Residential_load'].values/res_max).reshape(-1, 1)
util_res_norm_load = min_max_scaler.fit_transform(util_res_load_data)

util_com_load_data = (loads_df['Commercial_load'].values/com_max).reshape(-1, 1)
util_com_norm_load = min_max_scaler.fit_transform(util_com_load_data)

# Definition of a SAC Agent

In [ ]:
class GaussianPolicy(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim=64):
        super(GaussianPolicy, self).__init__()
        self.fc1 = nn.Linear(state_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.mean_layer = nn.Linear(hidden_dim, action_dim)
        self.log_std_layer = nn.Linear(hidden_dim, action_dim)
    
    def forward(self, state):
        x = F.relu(self.fc1(state))
        x = F.relu(self.fc2(x))
        mean = self.mean_layer(x)
        log_std = torch.clamp(self.log_std_layer(x), -20, 2)
        return mean, log_std

    def sample(self, state):
        mean, log_std = self.forward(state)
        std = log_std.exp()
        normal = torch.distributions.Normal(mean, std)
        z = normal.rsample()
        action = torch.tanh(z)
        log_prob = normal.log_prob(z) - torch.log(1 - action.pow(2) + 1e-6)
        log_prob = log_prob.sum(1, keepdim=True)
        return action, log_prob

class QNetwork(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim=64):
        super(QNetwork, self).__init__()
        self.fc1 = nn.Linear(state_dim + action_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, 1)

    def forward(self, state, action):
        x = torch.cat([state, action], dim=1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)


In [ ]:
class SACAgent:
    def __init__(self, state_dim, action_dim, lr=3e-4, gamma=0.99, tau=0.005, alpha=0.2, automatic_entropy_tuning=True):
        self.policy = GaussianPolicy(state_dim, action_dim).to(dev)
        self.q1 = QNetwork(state_dim, action_dim).to(dev)
        self.q2 = QNetwork(state_dim, action_dim).to(dev)
        self.q1_target = QNetwork(state_dim, action_dim).to(dev)
        self.q2_target = QNetwork(state_dim, action_dim).to(dev)
        self.q1_target.load_state_dict(self.q1.state_dict())
        self.q2_target.load_state_dict(self.q2.state_dict())

        self.policy_optim = optim.Adam(self.policy.parameters(), lr=lr)
        self.q1_optim = optim.Adam(self.q1.parameters(), lr=lr)
        self.q2_optim = optim.Adam(self.q2.parameters(), lr=lr)

        self.gamma = gamma
        self.tau = tau
        self.alpha = alpha
        self.automatic_entropy_tuning = automatic_entropy_tuning

        if self.automatic_entropy_tuning:
            self.target_entropy = -action_dim
            self.log_alpha = torch.zeros(1, requires_grad=True, device=dev)
            self.alpha_optim = optim.Adam([self.log_alpha], lr=lr)

        # Track metrics
        self.last_actor_loss = 0.0
        self.last_critic1_loss = 0.0
        self.last_critic2_loss = 0.0
        self.last_alpha_loss = 0.0

    def select_action(self, state, evaluate=False):
        state = torch.tensor(state, dtype=torch.float32, device=dev).unsqueeze(0)
        with torch.no_grad():
            if evaluate:
                mean, _ = self.policy(state)
                action = torch.tanh(mean)
            else:
                action, _ = self.policy.sample(state)
        return action.cpu().numpy()[0]

    def update(self, replay_buffer, batch_size=64):
        if len(replay_buffer) < batch_size:
            return

        batch = [replay_buffer[i] for i in np.random.choice(len(replay_buffer), batch_size, replace=False)]
        states, actions, rewards, next_states = zip(*batch)
        
        states = torch.tensor(states, dtype=torch.float32, device=dev)
        actions = torch.tensor(actions, dtype=torch.float32, device=dev)
        rewards = torch.tensor(rewards, dtype=torch.float32, device=dev).unsqueeze(1)
        next_states = torch.tensor(next_states, dtype=torch.float32, device=dev)

        with torch.no_grad():
            next_action, next_log_prob = self.policy.sample(next_states)
            q1_next = self.q1_target(next_states, next_action)
            q2_next = self.q2_target(next_states, next_action)
            q_next = torch.min(q1_next, q2_next) - self.alpha * next_log_prob
            q_target = rewards + self.gamma * q_next

        # Update Q-networks
        q1_pred = self.q1(states, actions)
        q2_pred = self.q2(states, actions)
        critic1_loss = F.mse_loss(q1_pred, q_target)
        critic2_loss = F.mse_loss(q2_pred, q_target)

        self.q1_optim.zero_grad()
        critic1_loss.backward()
        self.q1_optim.step()

        self.q2_optim.zero_grad()
        critic2_loss.backward()
        self.q2_optim.step()

        # Update policy
        new_action, log_prob = self.policy.sample(states)
        q1_new = self.q1(states, new_action)
        q2_new = self.q2(states, new_action)
        q_new = torch.min(q1_new, q2_new)
        actor_loss = (self.alpha * log_prob - q_new).mean()

        self.policy_optim.zero_grad()
        actor_loss.backward()
        self.policy_optim.step()

        # Adjust alpha
        if self.automatic_entropy_tuning:
            alpha_loss = -(self.log_alpha * (log_prob + self.target_entropy).detach()).mean()
            self.alpha_optim.zero_grad()
            alpha_loss.backward()
            self.alpha_optim.step()
            self.alpha = self.log_alpha.exp().item()
        else:
            alpha_loss = torch.tensor(0.)

        # Soft update targets
        for param, target_param in zip(self.q1.parameters(), self.q1_target.parameters()):
            target_param.data.copy_(self.tau * param.data + (1 - self.tau) * target_param.data)
        for param, target_param in zip(self.q2.parameters(), self.q2_target.parameters()):
            target_param.data.copy_(self.tau * param.data + (1 - self.tau) * target_param.data)

        # Save metrics
        self.last_actor_loss = actor_loss.item()
        self.last_critic1_loss = critic1_loss.item()
        self.last_critic2_loss = critic2_loss.item()
        self.last_alpha_loss = alpha_loss.item()


# Training the agent

In [ ]:
state_dim = 24
action_dim = 24
agent = SACAgent(state_dim=state_dim, action_dim=action_dim)
replay_buffer = deque()
num_episodes = 15000
update_every = 50
batch_size = 64

res_utils_diff = np.array(util_res_norm_load - util_com_norm_load)
res_utils_diff[res_utils_diff < 0] = 0 #replace negative entries with zeroes

rewards_per_episode = []
actor_losses, critic1_losses, critic2_losses, alpha_losses = [], [], [], []

for episode in tqdm(range(num_episodes), desc="Training Progress"):
    total_reward = 0
    for i in range(len(res_norm_load) - 24):
        state = res_norm_load[i:i+24].flatten()
        action = agent.select_action(state)
        reward = -np.abs(action - res_norm_load[i+1:i+25].flatten()).sum() # reward for peak-shaving and valley-filling
        # reward = -np.abs(action - res_utils_diff[i+1:i+25].flatten()).sum() # reward for inter-distribution networks load balancing
        # reward = -np.abs(action - res_norm_load[i+1:i+25].flatten()).sum() - (np.abs(action - res_utils_diff[i+1:i+25].flatten()).sum()) # reward for both
        total_reward += reward
        next_state = res_norm_load[i+1:i+25].flatten() if i + 25 < len(res_norm_load) else state
        replay_buffer.append((state, action, reward, next_state))

        if i % update_every == 0:
            for _ in range(10):
                agent.update(replay_buffer, batch_size=batch_size)

    rewards_per_episode.append(total_reward)
    actor_losses.append(agent.last_actor_loss)
    critic1_losses.append(agent.last_critic1_loss)
    critic2_losses.append(agent.last_critic2_loss)
    alpha_losses.append(agent.last_alpha_loss)

    if episode % 1500 == 0:
            print(f"Episode {episode}, Total Reward: {total_reward}")


In [ ]:
plt.plot(rewards_per_episode)
plt.xlabel("Episode")
plt.ylabel("Total Reward")
plt.title("SAC Rewards per Episode")
plt.show()


In [ ]:

plt.plot(actor_losses, label="Actor Loss")
plt.plot(critic1_losses, label="Critic1 Loss")
plt.plot(critic2_losses, label="Critic2 Loss")
plt.plot(alpha_losses, label="Alpha Loss")
plt.xlabel("Episode")
plt.ylabel("Loss")
plt.title("SAC Losses per Episode")
plt.legend()
plt.show()


In [ ]:
stats_df = pd.DataFrame({
    'SAC rewards': rewards_per_episode,
    'SAC actor loss': actor_losses,
    'SAC critic1 loss': critic1_losses,
    'SAC critic2 loss': critic2_losses,
    'SAC alpha loss': alpha_losses
})
stats_df.to_excel("sac_stats.xlsx", index = False)
print("Data exported to sac_stats.xlsx")